In [13]:
import torch

# Check if GPU or MPS is available
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"CUDA GPU is enabled: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("MPS GPU is enabled.")
else:
    raise OSError(
        "No GPU or MPS device found. Please check your environment and ensure GPU or MPS support is configured."
    )

CUDA GPU is enabled: NVIDIA GeForce GTX 1050 Ti


In [3]:
from docling.document_converter import DocumentConverter
from docling.chunking import HybridChunker
import ollama

c:\Users\krish\Work\projects\esg-disclosure-classifier\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [14]:
converter = DocumentConverter()
chunker = HybridChunker()

doc = converter.convert(r"../airbus_report_of_the_board_of_directors_2024.pdf").document

texts = [chunk.text for chunk in chunker.chunk(doc)]

2025-11-13 23:09:13,050 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2025-11-13 23:09:17,845 - INFO - Going to convert document batch...
2025-11-13 23:09:17,860 - INFO - Initializing pipeline for StandardPdfPipeline with options hash 44ae89a68fc272bc7889292e9b5a1bad
2025-11-13 23:09:18,999 - INFO - Loading plugin 'docling_defaults'
2025-11-13 23:09:19,008 - INFO - Registered picture descriptions: ['vlm', 'api']
2025-11-13 23:09:19,049 - INFO - Loading plugin 'docling_defaults'
2025-11-13 23:09:19,070 - INFO - Registered ocr engines: ['auto', 'easyocr', 'ocrmac', 'rapidocr', 'tesserocr', 'tesseract']
2025-11-13 23:09:19,348 - INFO - rapidocr cannot be used because onnxruntime is not installed.
2025-11-13 23:09:19,351 - INFO - easyocr cannot be used because it is not installed.
2025-11-13 23:09:20,851 - INFO - Accelerator device: 'cuda:0'
[INFO] 2025-11-13 23:09:20,903 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2025-11-13 23:09:20,995 [RapidOCR] download_file.py:60: 

time reduced to 13 mins with GPU

In [7]:
with open('converted_doc.txt', 'w', encoding='utf8') as file:
    file.writelines(texts)

In [15]:
def embed(text_chunk: str, model_name='mxbai-embed-large'):
    return ollama.embed(
        model=model_name,
        input=text_chunk
    )['embeddings'][0]

In [25]:
from pymilvus import Collection, connections, DataType, FieldSchema, CollectionSchema, utility
import os

MILVUS_HOST = os.getenv('MILVUS_HOST', "localhost")
MILVUS_PORT = os.getenv('MILVUS_PORT', '19530')
COLLECTION_NAME = os.getenv('COLLECTION_NAME', 'report_chunks')
EMBED_DIM = os.getenv('EMBED_DIM', 1024)   

test_embeddings = ollama.embed(
  model='nomic-embed-text:v1.5',
  input='Llamas are members of the camelid family',
)

EMBED_DIM = len(test_embeddings.embeddings[0])
print(EMBED_DIM)

model_name = 'nomic-embed-text:v1.5'

class MilvusManager:
    _instance = None
    _collection = None

    def __new__(cls):
        if cls._instance is None:
            cls._instance = super(MilvusManager, cls).__new__(cls)
            cls._instance._connect()
            cls._instance._init_collection()
        return cls._instance

    def _connect(self):
        '''Establish a single connection to Milvus.'''
        connections.connect("default", host=MILVUS_HOST, port=MILVUS_PORT)
        print(connections.list_connections())

    def _init_collection(self):
        '''Initialize or load collection schema once.'''
        fields = [
            FieldSchema(name="id", dtype=DataType.INT64, is_primary=True, auto_id=True),
            FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=EMBED_DIM),
            FieldSchema(name="text", dtype=DataType.VARCHAR, max_length=2000)
        ]
        schema = CollectionSchema(fields, description="Report text chunks with embeddings")

        try:
            self._collection = Collection(COLLECTION_NAME)
        except Exception:
            self._collection = Collection(name=COLLECTION_NAME, schema=schema)

        # Create index if not already present
        if not self._collection.has_index():
            self._collection.create_index(
                field_name="embedding",
                index_params={"metric_type": "COSINE", "index_type": "IVF_FLAT", "params": {"nlist": 128}}
            )

    def get_collection(self) -> Collection:
        '''Return the active collection object.'''
        return self._collection
    
    def semantic_search(self, query: str, top_k: int = 5):
        '''
        
        '''
        vector = embed(query, model_name)

        collection = self.get_collection()
        collection.load()

        results = collection.search(
            data=[vector],
            anns_field="embedding",
            param={"metric_type": "COSINE", "params": {"nprobe": 10}},
            limit=top_k,
            output_fields=["text"]
        )

        return ([hit.entity.get("text") for hit in results[0]], results)

2025-11-13 23:40:36,879 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"


768


In [17]:
model_name = 'nomic-embed-text:v1.5'

embeddings = [embed(chunk, model_name) for chunk in texts]

milvus = MilvusManager()
collection = milvus.get_collection()

data = [embeddings, texts]  
collection.insert(data)
collection.flush()

2025-11-13 23:30:29,738 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"
2025-11-13 23:30:30,156 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"
2025-11-13 23:30:30,493 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"
2025-11-13 23:30:30,826 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"
2025-11-13 23:30:31,158 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"
2025-11-13 23:30:31,299 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"
2025-11-13 23:30:31,665 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"
2025-11-13 23:30:31,800 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"
2025-11-13 23:30:31,932 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"
2025-11-13 23:30:32,040 - INFO - HTTP Request: POST http://127.0.0.1:1143

[('default', <pymilvus.client.grpc_handler.GrpcHandler object at 0x0000011160BEE660>)]


E5-5_10
37d:

37.

The undertaking shall disclose the following information on its total amount of waste from its own operations, in tonnes or kilogrammes:
(a) the total amount of waste generated ;

(b) the total amount by weight diverted from disposal, with a breakdown between hazardous waste and non-hazardous waste and a breakdown by the following recovery operation types:

i. preparation for reuse;

ii. recycling ; and

iii. other recovery operations.

(c) the amount by weight directed to disposal by waste treatment type and the total amount summing all three types, with a breakdown between hazardous waste and non-hazardous waste. The waste treatment types to be disclosed are:

i. incineration ;

ii. landfill; and

iii. other disposal operations;

(d) the total amount and percentage of non-recycled waste ( 91 ) . 


In [12]:
search_terms = [
    "Total waste generated and diverted from disposal by type and recovery operation",
    "Hazardous and non-hazardous waste breakdown by recycling, reuse, and recovery",
    "Waste directed to disposal by treatment type such as incineration, landfill, or other",
    "Total and percentage of non-recycled waste disclosure requirement",
    "Waste generation and treatment data reported in tonnes or kilograms under own operations",

]

search_terms_keywords = [
    "total waste generated tonnes kilograms",
    "hazardous non-hazardous waste recovery reuse recycling",
    "waste disposal incineration landfill other operations",
    "non-recycled waste total percentage",
    "waste treatment type diversion disposal breakdown",
    ]

In [29]:
milvus = MilvusManager()

[('default', <pymilvus.client.grpc_handler.GrpcHandler object at 0x0000011114AA47D0>)]


In [30]:
context = milvus.semantic_search(query="the total amount and percentage of non-recycled waste")

2025-11-13 23:41:24,226 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"


In [32]:
context[1]

data: [[{'id': 462151888724051222, 'distance': 0.7668567895889282, 'entity': {'text': '(pre-treatment / temporary status), Unit = tons. Of which, amount going to any other disposal operation - (pre-treatment / temporary status), 2024 = 7,168. Total amount of non hazardous waste directed to disposal, Unit = tons. Total amount of non hazardous waste directed to disposal, 2024 = 10,994. Of which, amount going to landfilling, Unit = tons. Of which, amount going to landfilling, 2024 = 5,955. Of which, amount going to incineration without energy recovery, Unit = tons. Of which, amount going to incineration without energy recovery, 2024 = 183.1. Of which, amount going to any other disposal operation - (pre-treatment / temporary status), Unit = tons. Of which, amount going to any other disposal operation - (pre-treatment / temporary status), 2024 = 4,856. Total amount of non-recycled waste, Unit = tons. Total amount of non-recycled waste, 2024 = 60,653. %of non-recycled waste, Unit = %. %of no

In [38]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

def extract_esrs_paragraphs(url: str) -> pd.DataFrame:
    """
    Extract paragraphs (main + appendix) from an ESRS XBRL HTML document,
    ignoring the table of contents or header sections.
    """

    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"Error fetching URL: {e}")
        return pd.DataFrame(columns=["paragraph_name", "text"])

    try:
        soup = BeautifulSoup(response.text, "html.parser")

        # Gather all text-bearing tags
        text_blocks = soup.find_all(['p', 'div', 'span', 'li', 'td', 'section'])

        data = []
        last_para = None
        seen_real_content = False  # to skip "Contents" part

        # Regex to detect paragraph IDs (e.g. 37, 37a, 37(b), AR12a, AG37)
        para_pattern = re.compile(r'^(?:(AR|AG)\s*)?(\d+\(?[a-zA-Z]?\)?)\s*(.*)')

        for block in text_blocks:
            text = " ".join(block.get_text(strip=True).split())
            if not text:
                continue

            # Skip table of contents or intro-like sections
            if not seen_real_content:
                if re.match(r'(?i)\b(contents?|index|table of contents)\b', text):
                    continue
                # Start only when we hit a real numbered paragraph
                if re.match(r'^(AR|AG)?\s*\d+', text):
                    seen_real_content = True
                else:
                    continue

            match = para_pattern.match(text)
            if match:
                prefix = match.group(1) or ""
                num = match.group(2).replace("(", "").replace(")", "")
                para_text = match.group(3).strip()

                para_id = f"{prefix}{num}".strip()
                last_para = para_id
                data.append((para_id, para_text))
            else:
                # Continuation of previous paragraph
                if last_para and data:
                    data[-1] = (data[-1][0], data[-1][1] + " " + text)

        df = pd.DataFrame(data, columns=["paragraph_name", "text"])
        df.drop_duplicates(inplace=True)
        df.reset_index(drop=True, inplace=True)
        return df

    except Exception as e:
        print(f"Error parsing HTML: {e}")
        return pd.DataFrame(columns=["paragraph_name", "text"])

In [39]:
url = "https://xbrl.efrag.org/e-esrs/esrs-set1-2023.html#d1e24283-3-1"
df = extract_esrs_paragraphs(url)
print(df.head(20))
print(f"\nExtracted {len(df)} paragraphs (including appendix).")

   paragraph_name                                               text
0               1                                                  .
1               1  . Categories of ESRS Standards, reporting area...
2               1                                                 .1
3               1                    .1 Categories of ESRS Standards
4               1                                                 .2
5               1  .2 Reporting areas and minimum content disclos...
6               1                                                 .3
7               1                            .3 Drafting conventions
8               2                                                  .
9               2       . Qualitative characteristics of information
10              3                                                  .
11              3  . Double materiality as the basis for sustaina...
12              3                                                 .1
13              3  .1 Stakeholders

In [37]:
df

,paragraph_name,text
0,1,.
1,1,". Categories of ESRS Standards, reporting area..."
2,1,.1
3,1,.1 Categories of ESRS Standards
4,1,.2
...,...,...
2069,38,(39)Directive 2013/34/EU of the European Parl...
2070,39,(40)ArticleÂ 3(1) of Directive 2008/98/EC on ...
2071,40,(41)ArticleÂ 4(1) of the Directive 2008/98/EC...
2072,41,(42)ArticleÂ 3(9) of the Directive 2008/98/EC...
